In [ ]:
# Vector database : ChromaDB 사용
!pip install chromadb sentence-transformers

In [ ]:
import chromadb
from chromadb import PersistentClient   # DB 클라이언트 객체(DB 접속 관리자 역할)
# !pip show chromadb
print(chromadb.__file__)

In [ ]:
client = PersistentClient(path=".chroma")
!ls -a

# 컬렉션 생성
collection = client.get_or_create_collection("test")

print("데이터 벡터화 후 저장 ---")
texts = ["Hello world", "Chroma is cool"]
ids = ["doc1", "doc2"]
metas = [{"source":"greeting"}, {"source":"statement"}]

# 현재 컬렉션에 설정된 임베딩 함수 가져오기
embedding_fn = collection._embedding_function   # 기본 내장 임베딩 모델:all-MiniLM-L6-v2 지원

embeddings = embedding_fn(texts)
print(embeddings[0][:5])
print(type(embeddings), len(embeddings), len(embeddings[0]))
print()
for i, vector in enumerate(embeddings):
    print(f'문서 : {texts[i]}')
    print(f'임베딩 값 5개만 : {vector[:5]}') # 384차원에 float32 벡터로 변환
    print(f'차원수 : {len(vector)}')
    print("-" * 30)

print('참고 : 두 문장 간 유사도 계산')
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
print(f'두 문장 간 코사인 유사도 {sim:.4f}')

# 컬렉션에 저장
collection.add(
    documents=texts,
    embeddings=embeddings,
    metadatas=metas,
    ids=ids
)
print(collection.count())


In [ ]:
# 저장된 문서 조회
results = collection.get(include=['documents', 'metadatas'])
print(results)

for doc, meta, id in zip(results['documents'], results['metadatas'], results['ids']):
    print(f'id : {id}')
    print(f'documents : {doc}')
    print(f'metadatas : {meta}')
    print(f'**' * 30)

print()
print('저장된 문서 id 목록 : ', collection.get()['ids'])

results_vec = collection.get(include=['embeddings'])
first_embedding = results_vec['embeddings'][0]
print('first_embedding : ', first_embedding[:5])
print('임베딩 벡터 차원 수 : ', len(first_embedding))

for id, emb in zip(results_vec['ids'], results_vec['embeddings']):
    print(f'id : {id}')
    print(f'embeddings(앞 5개) : {emb[:5]}')


In [ ]:
# 벡터 기반 검색
# 검색용 질문
query_text = "What's the status of chroma?"
# print(embedding_fn([query_text]))
# 검색 문장 임베딩
query_embedding = embedding_fn([query_text])[0]

# Chroma에서 유사 문서 검색
search_result = collection.query(
    query_embeddings=[query_embedding],  # 질문을 벡터화
    n_results=2,
    include=['documents', 'metadatas', 'distances']
)

for i, (doc, meta, dist) in enumerate(zip(
    search_result['documents'][0],
    search_result['metadatas'][0],
    search_result['distances'][0]
)):
    print(f'\n결과 {i + 1}')
    print(f' documents {doc}')
    print(f' metadatas {meta}')
    print(f' distances {dist}')
